# Aula $ 2 $ – Aritmética de Inteiros Usando Portas Lógicas

Este notebook é uma atividade prática para implementar a unidade aritmética de um processador. O objetivo final é construir funções que realizem a soma, subtração e multiplicação de inteiros utilizando somente as portas lógicas que implementamos na aula 1. Suas implementações já estão fornecidas abaixo (execute estas célula).

In [ ]:
def nand(a, b):
    """Porta NAND: a única cuja implementação utiliza
       explicitamente comandos do Python."""
    return int(not (a and b))

In [1]:
def not_gate(a):
    """Inverte o sinal de entrada."""
    return nand(a, a)

In [2]:
def and_gate(a, b):
    """Retorna 1 apenas se ambos argumentos forem 1."""
    return not_gate(nand(a, b))

In [4]:
def or_gate(a, b):
    """Retorna 1 se pelo menos um argumento for 1."""
    return nand(not_gate(a), not_gate(b))

In [5]:
def xor(a, b):
    """Retorna 1 se as entradas forem diferentes (ou exclusivo)."""
    return and_gate(or_gate(a, b), not_gate(and_gate(a, b)))

## $ \S 1 $ Somando bits individuais

Para somar números usando o sistema binário, começaremos analisando a soma de
apenas dois bits ($ a $ e $ b $). O resultado consiste em dois componentes: a
_soma_ (o bit de menor valor) e o _resto_ (o "vai um", ou bit de transporte).

Observe a tabela verdade da soma binária:

| $ a $ | $ b $ | soma | resto |
| :---: | :---: | :---: | :---: |
| $ 0 $ | $ 0 $ | $ 0 $ | $ 0 $ |
| $ 0 $ | $ 1 $ | $ 1 $ | $ 0 $ |
| $ 1 $ | $ 0 $ | $ 1 $ | $ 0 $ |
| $ 1 $ | $ 1 $ | $ 0 $ | $ 1 $ |

__Exercício 1:__ Implemente as funções `sum_bit` e `carry_bit` que calculam a
soma e o resto de um par de bits. Identifique qual porta lógica (das
implementadas anteriormente) corresponde a cada coluna da tabela acima.

In [ ]:
def sum_bit(a, b):
    """Retorna o bit de soma de a e b."""
    # TODO: Implementar
    pass

def carry_bit(a, b):
    """Retorna o bit de transporte (resto) da soma de a e b."""
    # TODO: Implementar
    pass

Um __meio-somador__ (ou _half-adder_, em inglês) combina as duas lógicas
anteriores em um único componente. Ele recebe dois bits e produz duas saídas
simultaneamente.

__Exercício 2:__ Implemente a função `half_adder` utilizando as funções
que você acabou de criar.

In [ ]:
def half_adder(a, b):
    """
    Recebe dois bits (a, b). Retorna a dupla (soma, carry).
    """
    # TODO: Implementar
    pass

In [ ]:
def test_half_adder():
    # Test cases: (A, B, expected_sum, expected_carry)
    cases = [
        (0, 0, 0, 0),
        (0, 1, 1, 0),
        (1, 0, 1, 0),
        (1, 1, 0, 1)
    ]

    for a, b, exp_s, exp_c in cases:
        s = sum_bit(a, b)
        c = carry_bit(a, b)
        
        success = (s == exp_s and c == exp_c)
        emoji = "✅" if success else "❌"
        msg = f"a={a}, b={b} | Soma obtida={s}, Resto={c}"
        
        if not success:
            msg += f" (Soma esperada={exp_s}, Resto={exp_c})"
            
        print(f"{emoji} {msg}")

test_half_adder()

Enquanto o meio-somador (half-adder) lida com apenas dois bits, um somador real
precisa ser capaz de somar três bits: os dois bits atuais (`a` e `b`) e o bit de
transporte que veio da coluna anterior (`c_in``).

__Exercício 3:__ Implemente um somador completo (`full_adder`).

_Dica:_ Você pode construir um somador completo conectando dois meio-somadores
e utilizando uma porta `or_gate` para consolidar o bit de transporte final.

In [ ]:
def full_adder(a, b, c_in):
    """
    Recebe três bits (a, b, carry_in). Retorna a dupla (soma, carry_out),
    onde soma = a + b + carry_in; e carry_out é o bit a ser carregado desta soma.
    """
    # TODO: Implementar utilizando half_adder e or_gate
    pass

In [ ]:
def test_full_adder():
    # Casos de teste: (A, B, Cin, soma_esperada, transporte_esperado)
    cases = [
        (0, 0, 0, 0, 0),
        (0, 0, 1, 1, 0),
        (0, 1, 0, 1, 0),
        (0, 1, 1, 0, 1),
        (1, 0, 0, 1, 0),
        (1, 0, 1, 0, 1),
        (1, 1, 0, 0, 1),
        (1, 1, 1, 1, 1)
    ]

    for a, b, cin, exp_s, exp_c in cases:
        s, c = full_adder(a, b, cin)
        
        success = (s == exp_s and c == exp_c)
        emoji = "✅" if success else "❌"
        msg = f"a={a}, b={b}, cin={cin} | Soma obtida={s}, Resto={c}"
        
        if not success:
            msg += f" (Soma esperada={exp_s}, Resto={exp_c})"
            
        print(f"{emoji} {msg}")

test_full_adder()

## $ \S 2 $ Representação binária de números

Historicamente, o sistema decimal se estabeleceu sobretudo pelo fato de
nós termos $ 10 $ dedos. Contudo, em computadores _tudo_ é representado
utilizando códigos binários. 

Tanto o sistema decimal quanto o binário são sistemas posicionais. Isto
significa que o valor de um algarismo depende da posição que ele ocupa.

__Exemplo:__ (Base $10$)
$$7904_{10} = 7 \times 10^3 + 9 \times 10^2 + 0 \times 10^1 + 4 \times 10^0\,.$$

__Exemplo:__ (Base $2$)
$$10110_2 = 1 \times 2^4 + 0 \times 2^3 + 1 \times 2^2 + 1 \times 2^1 + 0 \times 2^0 = 22_{10}$$

## $ \S 3 $ Somando números binários

Agora que temos um `full_adder` (somador completo para pares de bits), podemos
somar números representados por $ n $ bits concatenando vários chips
deste tipo. A lógica é idêntica à soma que fazemos no papel: começamos da
direita (bit menos significativo) para a esquerda, sempre passando o resto
"vai um" para a próxima coluna.

_Instruções:_ Implemente a função `add8`. Ela deve receber duas _listas_, cada
uma contendo $ 8 $ bits (por exemplo, `a = [1, 0, 0, 0, 1, 0, 1, 0]`), e retornar uma
lista de $ 8 $ bits com o resultado da soma. Aqui os bits são listados
do menos significativo ao mais significativo, de modo que a lista `a` acima
representa:
$$
a = 01010001_2\,.
$$

**Exercício 3:** Implemente `add8`. Receba duas listas de $ 8 $ bits e retorne o resultado da soma de $8$ bits.
Ignore o carry final do oitavo bit (overflow).

In [ ]:
def add8(a, b):
    """
    Soma dois números binários de 8 bits.
    a, b: listas de 8 inteiros (0 ou 1).
    Retorna: lista de 8 inteiros.
    """
    # Dica: Comece o loop do índice 0 (menos significativo)
    # até o de índice 7 (mais significativo).
    # Utilize a função `full_adder` implementada anteriormente

    resultado = [0] * 8
    carry = 0
    # TODO: Implementar a lógica de encadeamento (ripple carry)
    pass

## $ \S 4 $ Subtração e Complemento de Dois

Um sistema binário de $n$ bits pode codificar $2^n$ valores diferentes. Para
representar números com sinal (positivos e negativos), o método mais eficiente,
e utilizado quase exclusivamente nos computadores modernos, é o do **complemento de dois**. 

Em relação à representação para números não-negativos representados por $ n $
bits com que vínhamos trabalhando até agora, a única diferença é que o bit mais
significativo, que antes tinha peso $ 2^{n-1} $, passa a ter peso $ -2^{n-1} $.
Ou, dito de outra forma, nesta representação para _números com sinal_, um determinado
código binário de $ n $ bits, em vez de representar um número não-negativo $ x $,
representa $ x - 2^n $.

__Exemplo:__ Seqüência `10110`:

- **Sem sinal ($x$):**
$$2^4 + 2^2 + 2^1 = 16 + 4 + 2 = 22$$

- **Com sinal ($x - 2^5$):**
$$-2^4 + 2^2 + 2^1 = -16 + 4 + 2 = 22 - 2^5 $$

__Exemplo:__ Sequência `11111`

- **Sem sinal ($x$):**
$$2^4 + 2^3 + 2^2 + 2^1 + 2^0 = 16 + 8 + 4 + 2 + 1 = 31$$

- **Com sinal ($x - 2^5$):**
$$-2^4 + 2^3 + 2^2 + 2^1 + 2^0 = -16 + 8 + 4 + 2 + 1 = -1 = 31 - 2^5 $$

Observe que neste sistema para representação de números negativos:
* O código de qualquer número não negativo começa com $0$. Estes números vão de $ 0 $ a $ 2^{n - 1} - 1 $.
* O código de qualquer número negativo começa com $1$. Estes números vão de $ -2^{n- 1} $ a $ -1 $.

Por exemplo, com $ n = 4 $, a faixa de inteiros que pode ser representada vai de $ -2^3 = -8 $ a $ 2^3 - 1 = 7 $.

A maior vantagem deste método é que a subtração pode ser tratada como um caso especial de adição. Para calcular $A - B$, o computador na verdade calcula $A + (-B)$. 

Por sua vez, para obter a representação de $-x$ a partir de $x$, procedemos assim:
1. Invertem-se todos os bits de $x$ ($ 0 $ vira $ 1 $ e  $ 1 $ vira $ 0 $); em código, isto seria denotado por `~x`.
2. Adiciona-se $1$ ao resultado, obtendo-se `~x + 1`.

O motivo pelo qual isto funciona é o seguinte:   Considere o número $ {\sim} x $
obtido de $ x $ invertendo-se todos os seus bits (exatamente como no passo $ 1
$).  Somando $ x $ e $ {\sim}x $, obtemos o número $ 111\cdots 1_2 = - 1 $, em
que todos os bits são iguais a $ 1 $. Somando $ 1 $ a este resultado obtemos
$ 000\cdots 0_2 = 0 $, uma vez que toda a aritmética de números de $ n $ bits é
feita módulo $ 2^n $ (o $ n+1 $-ésimo bit, que seria $ 1 $, é descartado).
Logo deduzimos que $ -x = ({\sim} x) + 1 $, ou seja, é exatamente o resultado dos
passos $ 1 $ e $ 2 $ descritos acima.

**Exercício 5:** Implemente a função `negate8` que inverte o sinal de um número
de $8$ bits e, em seguida, a função `sub8` para realizar a subtração utilizando
o somador que você construiu anteriormente.

In [ ]:
def negate8(a):
    """
    Inverte o sinal de um número de 8 bits usando complemento de dois.
    """
    # TODO: Implementar utilizando a função add8 e not_gate
    pass

In [ ]:
# --- Teste da Função negate8 ---
# Testando a inversão de sinal: 1 vira -1
# 1 em binário: [0, 0, 0, 0, 0, 0, 0, 1]
# -1 em Complemento de Dois: [1, 1, 1, 1, 1, 1, 1, 1]

entrada_pos = [0, 0, 0, 0, 0, 0, 0, 1]
resultado_neg = negate8(entrada_pos)

print("🧪 Teste negate8:")
print(f"📥 Entrada:  {entrada_pos} (Decimal: 1)")
print(f"📤 Saída:    {resultado_neg} (Deve ser -1)")
print("-" * 30)

In [ ]:
def sub8(a, b):
    """
    Realiza a subtração a - b utilizando a lógica de soma com o negativo.
    Lógica: a - b = a + (-b). a e b são listas de 8 bits.
    """
    # TODO: Implementar utilizando add8 e negate8
    pass

In [ ]:
# --- Teste da Função sub8 ➖ ---
# Testando a subtração: 10 - 3 = 7

# 10 binário (bit menos significativo primeiro):
# [0, 1, 0, 1, 0, 0, 0, 0]  (2 + 8)

# 3 binário (bit menos significativo primeiro):
# [1, 1, 0, 0, 0, 0, 0, 0]  (1 + 2)

# 7 binário (bit menos significativo primeiro):
# [1, 1, 1, 0, 0, 0, 0, 0]  (1 + 2 + 4)

val_a = [0, 1, 0, 1, 0, 0, 0, 0] # 10
val_b = [1, 1, 0, 0, 0, 0, 0, 0] # 3
resultado_sub = sub8(val_a, val_b)

print("🧪 Teste sub8:")
print(f"Valor A:  {val_a} (10)")
print(f"Valor B:  {val_b} (3)")
print(f"A - B:    {resultado_sub} (Resultado esperado: 7)")

# Verificação visual rápida
sucesso = (resultado_sub == [1, 1, 1, 0, 0, 0, 0, 0])
print(f"\nStatus do teste: {'✔️ PASSOU' if sucesso else '❌ FALHOU'}")